In [4]:
# -*- coding: utf-8 -*-
"""
Comparative Analysis of ML Classifiers for Medical Diagnosis
Student: Dhrubo Biswas
Roll: 2471511
Dataset: Breast Cancer Wisconsin (Diagnostic) - provided as CSV
"""

# ==================== Phase A: Data Engineering ====================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_curve, auc, classification_report

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the dataset (assuming 'data.csv' is in the current directory)
df = pd.read_csv('data.csv')

# Display basic info
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# 2. Drop the 'id' column as it is not a feature
df.drop('id', axis=1, inplace=True)

# 3. Convert target 'diagnosis' to binary: Malignant (M) -> 1, Benign (B) -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# 4. Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# 5. Separate features (X) and target (y)
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# 6. Feature scaling using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 7. Correlation matrix: find top 5 features most correlated with target
correlations = df.corr()['diagnosis'].drop('diagnosis').abs()
top5_features = correlations.sort_values(ascending=False).head(5)
print("\nTop 5 features most correlated with target (absolute correlation):")
for feat, corr in top5_features.items():
    print(f"  {feat}: {corr:.4f}")

# 8. Train-test split (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# ==================== Phase B: Model Implementation ====================

# Define three classifiers
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)  # probability=True for ROC
}

results = {}  # store metrics and predictions

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)
    # Predict on test set
    y_pred = model.predict(X_test)
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'Model': model,
        'y_pred': y_pred
    }
    
    print(f"\n{name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

# Identify best performing model (by accuracy)
best_model_name = max(results, key=lambda x: results[x]['Accuracy'])
best_model = results[best_model_name]['Model']
best_y_pred = results[best_model_name]['y_pred']
print(f"\nBest performing model: {best_model_name} (Accuracy = {results[best_model_name]['Accuracy']:.4f})")

# ==================== Phase C: Visualization ====================

# 1. Bar chart comparing Accuracy, Precision, Recall of all three models
metrics_df = pd.DataFrame({
    model: [results[model]['Accuracy'], results[model]['Precision'], results[model]['Recall']]
    for model in results
}, index=['Accuracy', 'Precision', 'Recall'])

metrics_df.plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title('Model Performance Comparison', fontsize=14)
plt.ylabel('Score', fontsize=12)
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on bars
for i, metric in enumerate(metrics_df.index):
    for j, model in enumerate(metrics_df.columns):
        val = metrics_df.iloc[i, j]
        plt.text(j - 0.2 + i*0.2, val + 0.01, f'{val:.2f}', fontsize=9, ha='center')

plt.tight_layout()
plt.show()

# 2. Confusion Matrix Heatmap for the best-performing model
cm = confusion_matrix(y_test, best_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Print interpretation
tn, fp, fn, tp = cm.ravel()
print(f"Confusion Matrix details for {best_model_name}:")
print(f"True Negatives (TN): {tn}   |   False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}   |   True Positives (TP): {tp}")

# 3. ROC Curve for the best-performing model
# Get predicted probabilities for the positive class (malignant = 1)
y_proba = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'{best_model_name} (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAnalysis complete. The best model shows excellent discrimination (AUC > 0.99).")

Dataset shape: (569, 33)

First 5 rows:
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst 

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values